In [9]:
import pandas as pd
import glob
import os

# --- ÉTAPE 1 : Chargement du Master ---
print("Étape 1 : Lecture du fichier Master...")
if os.path.exists('dataset_final_backup.csv'):
    df_master = pd.read_csv('dataset_final_backup.csv', low_memory=False)
else:
    print("ERREUR : Fichier Master introuvable.")

# --- ÉTAPE 2 : Scan et traitement robuste ---
print("Étape 2 : Scan des fichiers...")
fichiers_dossier = glob.glob('*.csv')
fichiers_carac = [f for f in fichiers_dossier if 'carac' in f.lower()]

list_df_carac = []
for f in fichiers_carac:
    try:
        with open(f, 'r', encoding='latin-1') as file:
            header = file.readline()
            sep = ';' if ';' in header else ('\t' if '\t' in header else ',')
        
        df_temp = pd.read_csv(f, sep=sep, encoding='latin-1', low_memory=False, on_bad_lines='skip')
        df_temp.columns = df_temp.columns.str.replace('"', '').str.strip().str.lower()
        
        if 'accident_id' in df_temp.columns:
            df_temp = df_temp.rename(columns={'accident_id': 'num_acc'})
            
        # Nettoyage de l'année
        if 'an' in df_temp.columns:
            df_temp['an'] = pd.to_numeric(df_temp['an'], errors='coerce')
            df_temp.loc[df_temp['an'] < 100, 'an'] += 2000
            df_temp = df_temp.dropna(subset=['an'])
            df_temp['an'] = df_temp['an'].astype(int)
            
            # On garde 'an' impérativement
            cols_voulues = ['num_acc', 'dep', 'lat', 'long', 'an']
            cols_ok = [c for c in cols_voulues if c in df_temp.columns]
            list_df_carac.append(df_temp[cols_ok])
            print(f" Succès : {f}")
        else:
            print(f" ÉCHEC : Colonne 'an' introuvable dans {f}")
            
    except Exception as e:
        print(f" Erreur sur {f} : {e}")

# Combinaison
df_geo_all = pd.concat(list_df_carac, ignore_index=True)
df_geo_all = df_geo_all.drop_duplicates(subset=['num_acc'])

# --- ÉTAPE 3 : Fusion sécurisée ---
print("Étape 3 : Fusion de la géographie...")

# On supprime seulement les colonnes géographiques pour éviter les doublons
for col in ['dep', 'lat', 'long']:
    if col in df_master.columns: 
        df_master = df_master.drop(columns=[col])

# Fusion sur 'num_acc'
df_final = df_master.merge(df_geo_all, on='num_acc', how='left')

# Si 'an' n'est pas dans df_final, on la récupère depuis le fichier géo
if 'an' not in df_final.columns:
    df_final = df_final.rename(columns={'an_y': 'an'})

# --- ÉTAPE 4 : Labelisation ---
dict_grav = {1: "Indemne", 2: "Tué", 3: "Blessé hospitalisé", 4: "Blessé léger"}
if 'grav' in df_final.columns: 
    df_final['grav_label'] = df_final['grav'].map(dict_grav)

# --- ÉTAPE 5 : Sauvegarde ---
df_final.to_csv('dataset_final_backup.csv', index=False)
print("\n--- MISSION ACCOMPLIE ---")
print(f"Total lignes finales : {df_final.shape[0]}")
print("Années présentes dans le fichier :", sorted(df_final['an'].unique()))

Étape 1 : Lecture du fichier Master...
Étape 2 : Scan des fichiers...
 Succès : caract-2023.csv
 Succès : caract-2024.csv
 Succès : caracteristiques-2017.csv
 Succès : caracteristiques-2018.csv
 Succès : caracteristiques-2019.csv
 Succès : caracteristiques-2020.csv
 Succès : caracteristiques-2021.csv
 Succès : caracteristiques-2022.csv
 Succès : caracteristiques_2005.csv
 Succès : caracteristiques_2006.csv
 Succès : caracteristiques_2007.csv
 Succès : caracteristiques_2008.csv
 Succès : caracteristiques_2009.csv
 Succès : caracteristiques_2010.csv
 Succès : caracteristiques_2011.csv
 Succès : caracteristiques_2012.csv
 Succès : caracteristiques_2013.csv
 Succès : caracteristiques_2014.csv
 Succès : caracteristiques_2015.csv
 Succès : caracteristiques_2016.csv
Étape 3 : Fusion de la géographie...

--- MISSION ACCOMPLIE ---
Total lignes finales : 5621008
Années présentes dans le fichier : [np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.